In [2]:
import os
os.chdir("..")
for f in ['lr_09.pkl', 'mlp1_09.pt', 'mlp2_09.pt']:
    path = f"data/results/classifiers/{f}"
    print(f"{'✓' if os.path.exists(path) else '✗'} {path}")

✓ data/results/classifiers/lr_09.pkl
✓ data/results/classifiers/mlp1_09.pt
✓ data/results/classifiers/mlp2_09.pt


In [3]:
# -----------------------------------------------
# 12a: Extract False Positive Seeds
# Identifies false positives for each classifier,
# applies k-means clustering, and saves centroid-
# proximal seeds for error-driven generation.
# -----------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, ClassifierMixin
from torch.optim import AdamW

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------------------------
# Load embeddings and raw training sentences
# Embeddings for classification, sentences for
# saving as generation seeds
# -----------------------------------------------
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")

train_df = pd.read_parquet("data/processed/train.parquet")

print(f"Train: {X_train.shape} | Positives: {y_train.sum()}")
print(f"Train df: {len(train_df)} rows")
print(f"Columns: {train_df.columns.tolist()}")

Train: (69510, 384) | Positives: 674
Train df: 69510 rows
Columns: ['symbol', 'sector', 'quarter', 'sentence', 'label']


In [4]:
# -----------------------------------------------
# MLP architecture — must match Notebook 09
# definition exactly for weight loading to work
# -----------------------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=384, hidden_dims=[256], dropout=0.1):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 2))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class MLPClassifier(BaseEstimator, ClassifierMixin):
    """Minimal wrapper — only predict_proba needed here."""
    def __init__(self, hidden_dims=[256]):
        self.hidden_dims = hidden_dims
        self.classes_    = np.array([0, 1])

    def load_weights(self, path: str):
        self.model_ = MLP(input_dim=384, hidden_dims=self.hidden_dims)
        self.model_.load_state_dict(torch.load(path, map_location='cpu'))
        self.model_.eval()
        return self

    def predict_proba(self, X):
        with torch.no_grad():
            probs = torch.softmax(
                self.model_(torch.tensor(X, dtype=torch.float)), dim=1
            ).numpy()
        return probs

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# -----------------------------------------------
# Load all three classifiers from Notebook 09
# -----------------------------------------------
# LR — loaded directly from pickle
with open("data/results/classifiers/lr_09.pkl", "rb") as f:
    clf_lr = pickle.load(f)
print("LR loaded.")

# MLP-1 — reinstantiate wrapper, load weights
clf_mlp1 = MLPClassifier(hidden_dims=[256])
clf_mlp1.load_weights("data/results/classifiers/mlp1_09.pt")
print("MLP-1 loaded.")

# MLP-2 — reinstantiate wrapper, load weights
clf_mlp2 = MLPClassifier(hidden_dims=[256, 128])
clf_mlp2.load_weights("data/results/classifiers/mlp2_09.pt")
print("MLP-2 loaded.")

# Quick sanity check — score distribution on first 100 training examples
for name, clf in [('LR', clf_lr), ('MLP-1', clf_mlp1), ('MLP-2', clf_mlp2)]:
    scores = clf.predict_proba(X_train[:100])[:, 1]
    print(f"{name}: score mean={scores.mean():.3f} | max={scores.max():.3f}")

LR loaded.
MLP-1 loaded.
MLP-2 loaded.
LR: score mean=0.178 | max=0.976
MLP-1: score mean=0.041 | max=0.833
MLP-2: score mean=0.029 | max=0.944


In [5]:
# -----------------------------------------------
# False positive extraction and seed selection
# For each classifier:
#   1. Predict on full training set
#   2. Identify false positives (pred=1, true=0)
#   3. Apply k-means (k=200) on FP embeddings
#   4. Select sentence closest to each centroid
#   5. Save as seed parquet for generation
# -----------------------------------------------
from sklearn.metrics.pairwise import euclidean_distances

K = 200  # number of clusters — controls seed diversity
os.makedirs("data/synthetic", exist_ok=True)

def extract_fp_seeds(
    clf,
    X_train: np.ndarray,
    y_train: np.ndarray,
    train_df: pd.DataFrame,
    threshold: float,
    k: int = K,
    name: str = "classifier",
) -> pd.DataFrame:
    """
    Extract false positive seeds via k-means clustering.

    Args:
        clf:       Trained classifier with predict_proba
        X_train:   Training embeddings
        y_train:   Training labels
        train_df:  Raw training dataframe (for sentence/sector)
        threshold: Decision threshold from Notebook 09
        k:         Number of k-means clusters
        name:      Classifier name for logging

    Returns:
        DataFrame of seed sentences, one per cluster centroid
    """
    # Get predictions at Notebook 09 threshold
    scores = clf.predict_proba(X_train)[:, 1]
    preds  = (scores >= threshold).astype(int)

    # False positives: predicted positive, true label negative
    fp_mask = (preds == 1) & (y_train == 0)
    X_fp    = X_train[fp_mask]
    idx_fp  = np.where(fp_mask)[0]

    print(f"\n{name}: {fp_mask.sum()} false positives "
          f"({100*fp_mask.mean():.1f}% of train set)")

    if len(X_fp) < k:
        print(f"  Warning: fewer FPs ({len(X_fp)}) than clusters ({k}) "
              f"— using k={len(X_fp)}")
        k = len(X_fp)

    # K-means clustering on FP embeddings
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    kmeans.fit(X_fp)

    # For each centroid, find closest FP embedding
    dists   = euclidean_distances(kmeans.cluster_centers_, X_fp)
    closest = dists.argmin(axis=1)  # index into X_fp

    # Map back to original training indices
    seed_idx = idx_fp[closest]

    # Build seed dataframe
    seeds_df = train_df.iloc[seed_idx][['sentence', 'sector']].copy()
    seeds_df = seeds_df.drop_duplicates(subset='sentence').reset_index(drop=True)

    print(f"  Seeds selected: {len(seeds_df)} "
          f"(after dedup from {len(closest)})")

    return seeds_df


# Notebook 09 thresholds
thresholds = {'LR': 0.69, 'MLP-1': 0.69, 'MLP-2': 0.69}

# Extract seeds for each classifier
seeds = {}
for name, clf, t in zip(
    ['LR', 'MLP-1', 'MLP-2'],
    [clf_lr, clf_mlp1, clf_mlp2],
    [thresholds['LR'], thresholds['MLP-1'], thresholds['MLP-2']],
):
    seeds[name] = extract_fp_seeds(
        clf, X_train, y_train, train_df,
        threshold=t, k=K, name=name,
    )

# Save seed parquets
output_paths = {
    'LR':    "data/synthetic/error_seeds_lr.parquet",
    'MLP-1': "data/synthetic/error_seeds_mlp1.parquet",
    'MLP-2': "data/synthetic/error_seeds_mlp2.parquet",
}

for name, df in seeds.items():
    df.to_parquet(output_paths[name], index=False)
    print(f"\n{name} seeds saved to {output_paths[name]}")
    print(f"  Sample seeds:")
    for s in df['sentence'].head(3).tolist():
        print(f"    - {s[:100]}")


LR: 5083 false positives (7.3% of train set)
  Seeds selected: 199 (after dedup from 200)

MLP-1: 1134 false positives (1.6% of train set)
  Seeds selected: 200 (after dedup from 200)

MLP-2: 1473 false positives (2.1% of train set)
  Seeds selected: 200 (after dedup from 200)

LR seeds saved to data/synthetic/error_seeds_lr.parquet
  Sample seeds:
    - So while we've planned for those comp increases, it does take some time to work through our P&L.
    - We believe we are out in front of the market with some of our work on rates, but it is too early in 
    - So it's not an immediate one quarter thing, but we think over the long-term, we're going to feel goo

MLP-1 seeds saved to data/synthetic/error_seeds_mlp1.parquet
  Sample seeds:
    - 2024 gross margin is expected to set down slightly versus 2023 due to lower FX hedge gains and the r
    - We believe that the market opportunity over the long term is actually moving into our sweet spot, no
    - Putting it all together, we expec